## TC 3007B
### GPT 2

<br>

#### Activity 2,3: Code GPT2

#### Team Members:

- Ulises Orlando Carrizalez Lerín A01027715
- Mónica Monserrat Martínez Vásquez A01710965
- María José Soto Castro A01705840
- Tomás Pérez Vera A01028008
- Grant Nathaniel Keegan A01700753
- Bárbara Paola Alcántara Vega A01799609



<br>

- Objective:
    - To understand the Transformer architecture.
    - To code GPT 2.
    - To gain understanding of the LLMs' autoregresive nature..
    
<br>

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in [this video](https://youtu.be/51jq4wnHYaY)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  
  
- Evaluation Criteria

    - Code Readability and Comments (40%).
    - Traning a LM,  complete 'Train function' (30%).
    - Generating at least 10 sentences, comple 'Sample function' (30%).

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.




## Model Implementation

## Imports

In [1]:
!pip install transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 KB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 KB 14.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.7 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 KB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.7/791.7 KB 18.4 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 KB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 23.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 KB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.2/193.2 KB 11.3 MB

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import torch.optim as optim
from transformers import GPT2TokenizerFast
import requests


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Model's Configuration

This class defines the central configuration of the transformer model, storing all essential hyperparameters in a single location. It specifies the vocabulary size, maximum sequence length, embedding dimensions, number of transformer layers, attention heads, and dropout rate. This unified configuration facilitates consistency across all model components and allows for centralized adjustments, following the design pattern used in original GPT-2 implementations.

In [3]:
class Config:
    """
    Model configuration class for a GPT-2 style transformer.

    Params:
    
    vocab_size : int
        Size of the vocabulary (number of possible token IDs).
    max_seq_length : int
        Maximum sequence length the model can handle.
    embed_size : int
        Dimensionality of the token and positional embeddings.
    num_layers : int
        Number of transformer blocks in the GPT-2 architecture.
    num_heads : int
        Number of attention heads in the multi-head attention mechanism.
    dropout : float
        Dropout rate applied in attention and feed-forward layers.

    Notes:

    GPT-2 uses a decoder-only transformer architecture.  
    This configuration object stores hyperparameters needed
    across multiple modules (embeddings, attention, FFN, etc.).
    """
    def __init__(self, vocab_size = 50257, max_seq_length = 128, embed_size = 768, num_layers = 12,
                 num_heads = 12, dropout = 0.1):
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embed_size = embed_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout

### Self Attention Mechanism

This class implements the central multi-head self-attention mechanism that allows the model to process relationships between tokens in a sequence. Each head calculates attention using independent linear projections for queries (Q), keys (K), and values (V), applying a causal mask to prevent future tokens from influencing current predictions. The concatenation and final projection of the outputs from all heads allows different types of linguistic dependencies to be captured, forming the basis of the model's predictive capability.

In [4]:
class SelfAttention(nn.Module):
    """
    Model configuration class for a GPT-2 style transformer.

    Params:
    
    vocab_size : int
        Size of the vocabulary (number of possible token IDs).
    max_seq_length : int
        Maximum sequence length the model can handle.
    embed_size : int
        Dimensionality of the token and positional embeddings.
    num_layers : int
        Number of transformer blocks in the GPT-2 architecture.
    num_heads : int
        Number of attention heads in the multi-head attention mechanism.
    dropout : float
        Dropout rate applied in attention and feed-forward layers.

    Notes:

    GPT-2 uses a decoder-only transformer architecture.  
    This configuration object stores hyperparameters needed
    across multiple modules (embeddings, attention, FFN, etc.).
    """
    
    def __init__(self, config):
        super().__init__()
        assert config.embed_size % config.num_heads == 0, 'sizes not compatible'
        self.num_heads = config.num_heads
        self.head_dim = config.embed_size // config.num_heads
        
        # Linear projections for Q, K and V
        self.W_q = nn.Linear(config.embed_size, config.embed_size)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)

        # Output projection after concatenation
        self.output = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

        # Precomputed causal mask: shape (1, 1, T, T)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.max_seq_length, config.max_seq_length)
                      ).view(-1, 1, config.max_seq_length, config.max_seq_length)
        )
        
    def forward(self, x):
        batch, seq_length, embed_dim = x.size() # B, T, D
        # Each becomes (B, num_heads, T, head_dim) after reshape + transpose
        Q = self.W_q(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) #after trans B, numheads, T, head dim
        K = self.W_k(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        ## Each becomes (B, num_heads, T, head_dim) after reshape + transpose
        attn = (Q@K.transpose(-2, -1))/(self.head_dim**0.5) #B, numheads, T, T
        # Future positions become -inf → zero prob after softmax
        attn = attn.masked_fill(self.mask[:, :, :seq_length, :seq_length] == 0, float('-inf'))
        attn = F.softmax(attn, dim = -1)
        # Softmax over key dimension
        attn = self.dropout(attn)
        # Weighted sum of values
        scores = attn @ V #B, numheads, T, head_dim
        # Concatenate heads
        scores = scores.transpose(1, 2).contiguous().view(batch, seq_length, embed_dim) # Convert (B, num_heads, T, head_dim) → (B, T, embed_size)
        scores = self.output(scores)
        
        return self.dropout(scores)

### Position-wise Feed-Forward layer

This class applies independent nonlinear transformations to each position after the attention layer. It consists of two linear layers with 4x intermediate expansion and GELU activation, introducing universal approximation capability to the model. By processing each token individually but sharing parameters across all positions, the FFN complements attention by allowing complex transformations of learned representations.

In [5]:
class FFN(nn.Module):
    """
    Position-wise feed-forward neural network used inside each transformer block.

    Params:

    config : Config
        Contains embedding size and dropout values.

    Notes:

    The FFN is applied independently to each position (token)
    and serves to transform representations after attention.
    """

    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        """
        Applies the two-layer feed-forward network.

        Params:

        x : torch.Tensor
            Input tensor of shape (batch, seq_length, embed_dim).

        Returns:

        torch.Tensor
            Output tensor of identical shape.
        """
        
        x = self.fc2(self.gelu(self.fc1(x)))
        return self.dropout(x)

### Transformer Decoder-Only Block

It represents a fundamental processing unit in GPT-2, combining self-attention and feed-forward networks with residual connections and layer normalization. The two-stage architecture (attention followed by FFN) with skip connections allows deep networks to be trained while avoiding the problem of vanishing gradients. Each block progressively refines token representations, capturing complex dependencies at different levels of abstraction.

In [6]:
class Transformer(nn.Module):
    """
    Single GPT-2 transformer block.

    Structure:

    1. LayerNorm -> Multi-Head Self-Attention -> Residual
    2. LayerNorm -> Feed-Forward Network -> Residual

    Params:

    config : Coonfig
        Model hyperparameters.

    Notes:
    
    Each block refines token representations while preserving
    positional dependencies through self-attention.
    """
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embed_size)
        self.attention = SelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embed_size)
        self.mlp = FFN(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

### GPT2 Model

Assemble all components into a complete autoregressive language model, integrating token and positional embeddings, multiple transformer blocks, and a final prediction layer. The weight tying technique between input and output embeddings improves efficiency and stability during training. The model processes token sequences to generate probability distributions over the vocabulary, enabling prediction of subsequent tokens in text generation tasks.

In [7]:
class GPT2(nn.Module):
    """
    Minimal GPT-2 style language model.

    Components:

    - Token embeddings
    - Positional embeddings
    - N stacked transformer blocks
    - Final LayerNorm
    - Output logits via weight tying with token embeddings

    Params:

    config : Coonfig
        Configuration with model hyperparameters.

    Notes:
    
    The model predicts next-token logits for each position.
    Weight tying (token_embed.weight) reduces parameters and
    stabilizes training, following the original GPT-2 design.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        
        self.dropout = nn.Dropout(config.dropout)

         # Stack of transformer blocks
        self.transformers = nn.Sequential(*[Transformer(config) for _ in range(config.num_layers)])
        self.norm1 = nn.LayerNorm(config.embed_size)

    def forward(self, input_tokens):
        """
        Forward pass for GPT-2 language model.

        Params:
      
        input_tokens : torch.Tensor
            Tensor of token IDs with shape (batch, seq_length).

        Returns:
       
        torch.Tensor
            Logits of shape (batch, seq_length, vocab_size),
            representing next-token prediction probabilities.
        """
        
        batch, seq_length = input_tokens.size()
        pos = torch.arange(0, seq_length, dtype = torch.long, device = input_tokens.device).unsqueeze(0)
        x = self.token_embed(input_tokens) + self.pos_embed(pos)
        x = self.dropout(x)
        x = self.transformers(x)
        x = self.norm1(x)

        # Output logits using tied embeddings
        return x @ self.token_embed.weight.t()
        
        

## Training Model

### Getting Text Corpus

Getting text corpus from the Winnieh The Pooh story from the Gutenberg Project website

In [8]:
url = "https://www.gutenberg.org/cache/epub/67098/pg67098.txt"
response = requests.get(url)
text = response.text

In [9]:
from transformers import GPT2TokenizerFast

### Corpus Tokenization

In [10]:
tokeniser = GPT2TokenizerFast.from_pretrained("gpt2")
tokeniser.pad_token = tokeniser.eos_token

tokens = tokeniser.encode(text)
data = torch.tensor(tokens, dtype=torch.long)

Token indices sequence length is longer than the specified maximum sequence length for this model (48955 > 1024). Running this sequence through the model will result in indexing errors


### Sequence Length Defintion

In [11]:
SEQ_LENTGH = 128

### Winnie The Pooh Custom Dataset

This class implements a custom dataset that generates pairs of input and target sequences shifted by one token, preparing the data for training in autoregressive language modeling. For each position in the tokenized text, it extracts a fixed-length sequence as input and the same sequence shifted by one token as target, allowing the model to learn to predict the next token at each position in the sequence. The design facilitates efficient training using PyTorch DataLoaders without the need for padding.


In [12]:
class WinniePooh(Dataset):
    """
    PyTorch Dataset for creating sequential training samples
    from a text corpus such as the Winnie-the-Pooh dataset.

    This dataset generates input-target pairs suitable for
    autoregressive language modeling.  
    For each position `idx`, it returns:

        x = text[idx : idx + seq_length]
        y = text[idx + 1 : idx + seq_length + 1]

    which effectively shifts the sequence by one token.  
    The model learns to predict token y[t] given x[t].

    Params:
    
    data : list[int] or torch.Tensor
        Tokenized text data (a sequence of token IDs).
    seq_length : int
        Length of each input sequence fed to the model.

    Notes:
    
    - This dataset does not perform padding; it assumes every
      extracted sequence has `seq_length` tokens.
    - It supports indexing and length operations so it can be
      wrapped in a DataLoader for batching and shuffling.
    """
    def __init__(self, data, seq_length):
        self.text = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.text) - self.seq_length

    def __getitem__(self, idx):
        x = self.text[idx: idx + self.seq_length]
        y = self.text[idx + 1: idx + self.seq_length + 1]
        return x, y

### Winnie The Pooh Dataset and DataLoader Instance 

In [13]:
winnie_pooh = WinniePooh(data, SEQ_LENTGH)
winnie_pooh_loader = DataLoader(winnie_pooh, batch_size=16, shuffle= True)

### Model's Initialization and Hardware Configuration

In [14]:
config = Config(
    vocab_size = tokeniser.vocab_size,
    max_seq_length = SEQ_LENTGH, 
    embed_size = 128, 
    num_layers = 4,
    num_heads = 4,
    dropout = 0.1
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2(config).to(device)

### Optimiser Configuration

In [15]:
optimiser = optim.Adam(model.parameters(), lr=3e-4)

### Model's Training Function

This function implements the complete training cycle for the GPT-2 model, running multiple epochs on the dataset. In each iteration, it processes batches of sequences, calculates the cross-entropy loss between the model's predictions and the shifted target tokens, and updates the parameters using backpropagation. Training follows the standard autoregressive approach where the model learns to predict the next token in the sequence, with periodic monitoring of progress by printing intermediate losses.

In [16]:
def train(model, loader, optimiser, epochs = 30):
    """
    Train the GPT-like model using autoregressive language modeling.

    This function iterates over the dataset for a specified number
    of epochs, computes the cross-entropy loss between the model
    predictions and the target shifted sequence, performs
    backpropagation, and updates the model parameters.

    Params:
  
    model : torch.nn.Module
        The GPT model being trained.
    loader : torch.utils.data.DataLoader
        DataLoader that provides batches of (input, target) pairs.
    optimiser : torch.optim.Optimizer
        Optimizer used to update the model parameters.
    epochs : int, optional
        Number of training epochs (default is 3).

    Notes:
    
    - Uses causal (autoregressive) training: model predicts token y[t]
      given sequence x up to t.
    - Loss is computed as cross-entropy over all tokens in the batch.
    - Prints intermediate losses every 500 steps.
    """
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)
            optimiser.zero_grad()
            scores = model(x) #shape is (bminibatch, seq_lenght, vocab_size)
            loss = F.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1)) #y is shape (B, T)
            loss.backward()
            optimiser.step()

            total_loss += loss.item()

            if i%500 == 0:
                print(f'epoch: {epoch+1}, step: {i}, loss:{loss.item():.4f}')

        epoch_loss = total_loss / len(loader)
        print(f'epoch: {epoch + 1}. Loss: {epoch_loss:.4f}')


### Text Generation Function 

This function implements autoregressive text generation using the trained GPT-2 model. It begins by encoding an initial prompt and then iterates by generating one token at a time, always keeping the context window within the model's length limits. The temperature controls the randomness in the sampling, allowing you to adjust the balance between creativity and coherence in the generated text. The process continues until the specified length is reached, finally returning the complete decoded text.

In [17]:
def sample(model, device, tokenizer, prompt, length=50, temperature=1.0):
    """
    Generate text from a trained GPT model using sampling.

    Given a text prompt, the model autoregressively predicts the
    next token one step at a time, appends it to the context, and
    repeats until the desired sequence length is produced.

    Parameters:
    
    model : torch.nn.Module
        Trained GPT model used for text generation.
    device : torch.device
        Device where inference is executed (CPU or GPU).
    tokenizer : transformers-like tokenizer
        Tokenizer providing `encode()` and `decode()` methods.
    prompt : str
        Initial text prompt used to seed the generation process.
    length : int, optional
        Number of tokens to generate beyond the prompt.
    temperature : float, optional
        Sampling temperature:
        - higher (>1.0) → more randomness
        - lower (<1.0) → more deterministic

    Returns:
    
    str
        The generated text decoded from token IDs.

    Notes:
    
    - Sampling uses multinomial sampling over the softmax distribution.
    - The context is truncated to SEQ_LENGTH tokens before each inference
      step to respect the model's maximum sequence size.
    """

    model.eval()
    tokens = tokenizer.encode(prompt, return_tensors='pt').to(device)

    for _ in range(length):
        tokens_cond = tokens[:, -SEQ_LENTGH:]  # last window of tokens

        with torch.no_grad():
            logits = model(tokens_cond)

        # Temperature controls randomness
        next_token_logits = logits[:, -1, :] / temperature

        # Sample from the probability distribution
        next_token = torch.multinomial(
            F.softmax(next_token_logits, dim=-1),
            num_samples=1
        )

        tokens = torch.cat([tokens, next_token], dim=1)

    print(tokens)
    return tokenizer.decode(tokens[0])

### Text Generation Before Training

In [18]:
print(sample(model, device, tokeniser, prompt=" The fantasy world", length=50))

tensor([[ 383, 8842,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,
          995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,
          995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,
          995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,  995,
          995,  995,  995,  995,  995]], device='cuda:0')
 The fantasy world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world world


In [19]:
print(sample(model, device, tokeniser, prompt="A bear eating honey", length=50))

tensor([[   32,  6842,  6600, 12498, 12498, 12498, 12498, 12498, 12498, 12498,
         12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498,
         12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498,
         12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498,
         12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498, 12498,
         12498, 12498, 12498, 12498]], device='cuda:0')
A bear eating honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey honey


In [20]:
print(sample(model, device, tokeniser, prompt="The tiger is jumping in the park", length=50))

tensor([[  464, 26241,   318, 14284,   287,   262,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952]], device='cuda:0')
The tiger is jumping in the park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park


In [21]:
print(sample(model, device, tokeniser, prompt="Dancing under the rain", length=50))

tensor([[  35, 5077,  739,  262, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290,
         6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290,
         6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290,
         6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290, 6290,
         6290, 6290, 6290, 6290, 6290, 6290, 6290]], device='cuda:0')
Dancing under the rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain rain


In [22]:
print(sample(model, device, tokeniser, prompt="Kids playing in the park", length=50))

tensor([[40229,  2712,   287,   262,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,  3952,
          3952,  3952,  3952,  3952,  3952]], device='cuda:0')
Kids playing in the park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park park


In [23]:
print(sample(model, device, tokeniser, prompt="A big red balloon", length=50))

tensor([[   32,  1263,  2266, 21190, 21190, 21190, 21190, 21190, 21190, 21190,
         21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190,
         21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190,
         21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190,
         21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190, 21190,
         21190, 21190, 21190, 21190]], device='cuda:0')
A big red balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon balloon


In [24]:
print(sample(model, device, tokeniser, prompt="A bunch of sweet candies", length=50))

tensor([[  32, 7684,  286, 6029, 2658,  444,  444,  444,  444,  444,  444,  444,
          444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,
          444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,
          444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,  444,
          444,  444,  444,  444,  444,  444,  444,  444]], device='cuda:0')
A bunch of sweet candiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesiesies


In [25]:
print(sample(model, device, tokeniser, prompt="A man playing guitar", length=50))

tensor([[   32,   582,  2712, 10047, 10047, 10047, 10047, 10047, 10047, 10047,
         10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047,
         10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047,
         10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047,
         10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047, 10047,
         10047, 10047, 10047, 10047]], device='cuda:0')
A man playing guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar guitar


In [26]:
print(sample(model, device, tokeniser, prompt="Flying over the sea", length=50))

tensor([[49095,   625,   262,  5417,  5417,  5417,  5417,  5417,  5417,  5417,
          5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,
          5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,
          5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,
          5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,  5417,
          5417,  5417,  5417,  5417]], device='cuda:0')
Flying over the sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea sea


In [27]:
print(sample(model, device, tokeniser, prompt="A girl singing", length=50))

tensor([[   32,  2576, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777,
         13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777,
         13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777,
         13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777,
         13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777, 13777,
         13777, 13777, 13777]], device='cuda:0')
A girl singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing singing


### Training Model

In [28]:
train(model, winnie_pooh_loader, optimiser)

epoch: 1, step: 0, loss:76.8546
epoch: 1, step: 500, loss:7.8597
epoch: 1, step: 1000, loss:5.4975
epoch: 1, step: 1500, loss:5.4965
epoch: 1, step: 2000, loss:5.4138
epoch: 1, step: 2500, loss:4.4869
epoch: 1, step: 3000, loss:4.5383
epoch: 1. Loss: 7.0060
epoch: 2, step: 0, loss:4.4429
epoch: 2, step: 500, loss:3.8846
epoch: 2, step: 1000, loss:4.0988
epoch: 2, step: 1500, loss:4.1267
epoch: 2, step: 2000, loss:3.9381
epoch: 2, step: 2500, loss:3.7068
epoch: 2, step: 3000, loss:3.5791
epoch: 2. Loss: 3.8798
epoch: 3, step: 0, loss:3.1843
epoch: 3, step: 500, loss:3.4923
epoch: 3, step: 1000, loss:3.2173
epoch: 3, step: 1500, loss:2.9656
epoch: 3, step: 2000, loss:2.8022
epoch: 3, step: 2500, loss:2.7947
epoch: 3, step: 3000, loss:2.4801
epoch: 3. Loss: 2.8979
epoch: 4, step: 0, loss:2.4836
epoch: 4, step: 500, loss:2.3512
epoch: 4, step: 1000, loss:2.3162
epoch: 4, step: 1500, loss:1.9190
epoch: 4, step: 2000, loss:2.0901
epoch: 4, step: 2500, loss:1.8468
epoch: 4, step: 3000, loss:1

### Text Generation After Training

In [29]:
print(sample(model, device, tokeniser, prompt=" The fantasy world", length=50))

tensor([[  383,  8842,   995,   379,   262,  6766,    11,   290,  8848,   656,
           262,  1660,   290, 37516,    12,   325, 20987,   679,   487,   282,
           931,   201,   198,  9776, 43390,   663, 39940,    11,   290,  2282,
           284,  2346,    11,   366, 16371,   922, 12498,   428,    11,   314,
           201,   198,  9099,   470,   760,   618,   314,  1053, 29187,  1365,
           553,  7695,  1219]], device='cuda:0')
 The fantasy world at the sky, and boat into the water and eighty-seventh Heffalump
was licking its jaws, and saying to itself, "Very good honey this, I
don't know when I've tasted better," Pooh


In [30]:
print(sample(model, device, tokeniser, prompt="A bear eating honey", length=50))

tensor([[   32,  6842,  6600, 12498,   553,   339,  1807,    13,   366,  4366,
           460,   470,    13,   201,   198,  2504,   338,   703,   340,   318,
           526,   201,   198,   201,   198,  1537,   612,   547,  7188,   618,
         23097,  1616, 16555,   326,   509, 16484,  3521,   470,    13, 18023,
            11,   201,   198, 12518,   339,   550,   257,   890,  2513,  1363,
           832,   262,  9115,    11]], device='cuda:0')
A bear eating honey," he thought. "Some can't.
That's how it is."

But there were moments when Piglet wished that Kanga couldn't. Often,
when he had a long walk home through the Forest,


In [31]:
print(sample(model, device, tokeniser, prompt="The tiger is jumping in the park", length=50))

tensor([[  464, 26241,   318, 14284,   287,   262,  3952,    13,   201,   198,
           201,   198,     1,  2504,   338,  2089,   553,   531,   412,  2959,
           382, 30338,   813,    13,   366,  1026,  5905,  1299,   201,   198,
         19693,    13,  1400, 12902,   526,   201,   198,   201,   198,     1,
          1639,  1276,   423,  1364,   340,  7382,   553,   531, 19454,   494,
            12,  1169,    12, 18833,  1219,    13,   201]], device='cuda:0')
The tiger is jumping in the park.

"That's bad," said Eeyore gloomily. "It Explains
Everything. No Wonder."

"You must have left it somewhere," said Winnie-the-Pooh.


In [32]:
print(sample(model, device, tokeniser, prompt="Dancing under the rain", length=50))

tensor([[   35,  5077,   739,   262,  6290,    13,   201,   198,   201,   198,
            16,    13,    37,    13,    21,    13, 24413,  3620,    45,  9050,
           532,   921,  4236,   284, 47846,  1958,   290,  1745,   262,  5693,
            11,   262,   201,   198,  2213, 36920,   668,  4870,    11,   597,
          5797,   393,  6538,   286,   262,  5693,    11,  2687,   201,   198,
         15234,  2530,  9088,   286,  4935]], device='cuda:0')
Dancing under the rain.

1.F.6. INDEMNITY - You agree to indemnify and hold the Foundation, the
trademark owner, any agent or employee of the Foundation, anyone
providing copies of Project


In [33]:
print(sample(model, device, tokeniser, prompt="Kids playing in the park", length=50))

tensor([[40229,  2712,   287,   262,  3952,   510,   284,   465,  1545, 12803,
         12325,    13,   201,   198,   201,   198,     1,  1639,   460,   470,
           307,   287,  3576,   329,   890,  1231,  1223, 14802,    68,    13,
          1318,   389,   617,   201,   198, 15332,   508,  2221,   262, 21980,
           379,   262,  3726,    11,  1444, 34882,  1268,    11,   290,  2513,
           355,   201,   198, 24209,   306]], device='cuda:0')
Kids playing in the park up to his friend Christopher Robin.

"You can't be in London for long without something bravee. There are some
people who begin the Zoo at the beginning, called WAYIN, and walk as
quickly


In [34]:
print(sample(model, device, tokeniser, prompt="A big red balloon", length=50))

tensor([[   32,  1263,  2266, 21190,    30,    62,     1,   201,   198,   201,
           198,     1,  5812,     0,  3894,    11,  4650,  3772,  5860,   286,
           262,  1110,    11,   412,  2959,   382,   526,   201,   198,   201,
           198,     1,  1870,   867,  3772,  5860,   284,   345,    11,  7695,
          1219, 14732,   526,   201,   198,   201,   198,     1,  1537,   340,
          2125,   470,  4808,  1820]], device='cuda:0')
A big red balloon?_"

"Oh! Well, Many happy returns of the day, Eeyore."

"And many happy returns to you, Pooh Bear."

"But it isn't _my


In [35]:
print(sample(model, device, tokeniser, prompt="A bunch of sweet candies", length=50))

tensor([[   32,  7684,   286,  6029,  2658,   444,   286,   201,   198,   220,
           220,   220,   220,   314,   910,    11,   340,  2081,   201,   198,
           220,   220,   314,   716,  3375,   286,  7695,  1219,   438,   201,
           198,   220,   220,   220,   220, 44104,  5189,   508,    30,    62,
             8,   201,   198,   220,   220,  3226,  7695,  1219,     0,   201,
           198,   220,   220,   220, 44104,    40]], device='cuda:0')
A bunch of sweet candies of
     I say, it true
   I am talking of Pooh--
     (_Of who?_)
   Of Pooh!
    (_I


In [36]:
print(sample(model, device, tokeniser, prompt="A man playing guitar", length=50))

tensor([[   32,   582,  2712, 10047,   287,   616,  2951,    11,   314,  3521,
           470,   201,   198,   961,  1243,    11,   523,   314,  3181,   340,
           284,   345,    13,  1550,   616,  8848,   526,   201,   198,   201,
           198,  3152,   777,  6613,  2456,   339,  2921, 12803, 12325,   262,
          2051,   496,    13,   201,   198,   201,   198,     1,  1537,   340,
           338,   422, 23097,  1616]], device='cuda:0')
A man playing guitar in my eyes, I couldn't
read things, so I brought it to you. On my boat."

With these proud words he gave Christopher Robin the missage.

"But it's from Piglet


In [37]:
print(sample(model, device, tokeniser, prompt="Flying over the sea", length=50))

tensor([[49095,   625,   262,  5417,   284,   201,   198,   411, 15509,   683,
            13, 12803, 12325,   290,  7695,  1219,   757,  1106,   201,   198,
           201,   198,  1870,   326,   318,  1107,   262,   886,   286,   262,
          1621,    11,   290,   314,   716,   845, 10032,   706,   326,   201,
           198, 12957,  6827,    11,   314,   892,   314,  2236,  2245,   612,
            13,   201,   198,   201]], device='cuda:0')
Flying over the sea to
rescue him. Christopher Robin and Pooh again....

And that is really the end of the story, and I am very tired after that
last sentence, I think I shall stop there.



In [38]:
print(sample(model, device, tokeniser, prompt="A girl singing", length=50))

tensor([[   32,  2576, 13777, 14801,   371,  2238,  8348,   356,   910,    11,
         13674,  2474,   201,   198,   201,   198,     1,    62,  2437,   466,
           553,   531,  7695,  1219,    13,   201,   198,   201,   198,     1,
           464,  1266,   835,   553,   531, 25498,    11,   366, 19188,   307,
           428,    13,   383,  1266,   835,   561,   307,   284,   201,   198,
           301,  2287, 14801]], device='cuda:0')
A girl singing Baby Roo?' we say, dear!"

"_How do," said Pooh.

"The best way," said Rabbit, "would be this. The best way would be to
steal Baby
